In [ ]:
from pathlib import Path
import sys
from datetime import date
import pandas as pd
import gc
import os
import glob
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import logging

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s"
)

# --- Paths / imports -------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
PREPROCESSING_DIR = PROJECT_ROOT / "functions" / "preprocessing"
for p in (PROJECT_ROOT, PREPROCESSING_DIR):
    if str(p) not in sys.path:
        sys.path.append(str(p))

from server_config import (
    datapath,
    proj_sheet,
    preprocessed_path,
    raw_path,
    backup_path,
    preprocessed_path_freezed,
)
from missing_data import compute_availability_metrics

# --- Dates ------------------------------------------------------------
today_str = date.today().strftime("%d%m%Y")
today_day = pd.Timestamp.today().normalize()
# today_str = "25082025"

# --- Path -------------------------------------------------------------

datapath = Path(raw_path) / f"export_tiki_{today_str}"

In [ ]:
# df_backup_recent = pd.read_feather(preprocessed_path + "/backup_passive_recent.feather", dtype_backend="pyarrow")
df_backup_recent = pd.read_feather("temp_df_backup_recent.feather")
# df_backup_recent = pd.read_feather("temp_df_backup_recent.feather", dtype_backend="pyarrow")
df_backup_recent.head()

In [ ]:
df_backup_recent.memory_usage(deep=True) / 1024**2

In [ ]:
df_backup_recent.memory_usage(deep=True).sum() / 1024**2

In [ ]:
df_backup_recent.dtypes

## generation & trustworthiness stuff (redo at some point)

In [ ]:
# Create a series with MultiIndex (customer, start_day) and unique timezoneOffset values
timezone_series = (
    df_complete.groupby(["customer", "start_day"])["timezoneOffset"]
    .apply(lambda x: x.dropna().unique())
    .reset_index()
)

# Convert to a proper series with MultiIndex
timezone_series_indexed = timezone_series.set_index(["customer", "start_day"])[
    "timezoneOffset"
]

print(f"Series shape: {timezone_series_indexed.shape}")
print(
    f"Number of unique customer-start_day combinations: {len(timezone_series_indexed)}"
)
print("\nFirst 10 entries:")
print(timezone_series_indexed.head(10))

In [ ]:
# Analyze the timezone offsets
print("Analysis of timezoneOffset patterns:")
print("=" * 50)

# Check how many unique timezone offsets each customer-day combination has
offset_counts = timezone_series_indexed.apply(len)
print(f"Distribution of number of unique offsets per customer-day:")
print(offset_counts.value_counts().sort_index())

# Find cases with multiple timezone offsets on the same day
multiple_offsets = timezone_series_indexed[offset_counts > 1]
print(
    f"\nCustomer-day combinations with multiple timezone offsets: {len(multiple_offsets)}"
)

if len(multiple_offsets) > 0:
    print("\nFirst few examples of multiple offsets per day:")
    for i, (idx, berlin_offsets) in enumerate(multiple_offsets.head(5).items()):
        customer, day = idx
        print(f"  {customer} on {day.date()}: {berlin_offsets}")

# Show some statistics about timezone offsets
all_offsets = df_complete["timezoneOffset"].dropna()
print(f"\nOverall timezone offset statistics:")
print(f"  Total non-null timezone offset records: {len(all_offsets):,}")
print(f"  Unique timezone offset values: {sorted(all_offsets.unique())}")
print(
    f"  Most common offset: {all_offsets.mode().iloc[0]} (appears {(all_offsets == all_offsets.mode().iloc[0]).sum():,} times)"
)

# Store the final series for easy access
print(f"\nFinal series 'timezone_series_indexed' created with:")
print(f"  - MultiIndex: (customer, start_day)")
print(f"  - Values: unique timezoneOffset arrays for each customer-day")
print(f"  - Shape: {timezone_series_indexed.shape}")

In [ ]:
c, d = timezone_series_indexed[offset_counts >= 3].index[2]
# df_complete[(df_complete["customer"] == c) & (df_complete["start_day"] == d)].groupby("type")["timezoneOffset"].unique()
df_cd = df_backup_recent[
    (df_backup_recent["customer"] == c) & (df_complete["start_day"] == d)
]
df_cd
df_cd["generation"].apply(lambda x: generation_type_map[x])

#### generatioin & trustworthiness

generation:

| Parameter | Description |
|-----------|-------------|
| manual_entry | Data was manually entered by a user |
| manual_measurement | Data was recorded by a sensor measurement manually triggered by a user |
| automated_measurement | Data was recorded by a passive sensor measurement |
| ~~smartphone~~ | Data was recorded by a smartphone sensor |
| tracker | Data was recorded by a wearable sensor or medical device |
| third_party | Data was recorded by a third-party app |
| calculation | Data was calculated by Thryve (e.g., daily steps if not provided by manufacturer)


trustworthiness:

| Parameter | Description |
|-----------|-------------|
| **_unfavorable_measurement_context_** | Device manufacturer believes recording was made under suboptimal conditions (e.g., during movement when stillness is recommended) |
| **_doubt_from_device_source_** | Device algorithms flagged the measurement as potentially unreliable |
| ~~doubt_from_user~~ | User manually tagged the recording as unlikely or implausible |
| **_verified_from_device_source_** | Device manufacturer considers the recording plausible and reliable |
| ~~verified_from_user~~ | User manually verified |



In [ ]:
print("generation:", df_complete.generation.unique())
print("trustworthiness:", df_complete.trustworthiness.unique())
print("medicalGrade:", df_complete.medicalGrade.unique())
print("userReliability:", df_complete.userReliability.unique())
print("chronologicalExactness:", df_complete.chronologicalExactness.unique())

In [ ]:
generation_type_map = {
    -1: "unknown",
    10: "manual_entry",
    20: "manual_measurement",
    30: "automated_measurement",
    40: "calculation",
    50: "smartphone",  # not present
    60: "tracker",
    70: "third_party",
}

trustworthiness_type_map = {
    -1: "unknown",
    10: "plausible",
    20: "verified_from_device_source",  # present
    30: "verified_from_user",
    40: "verified_from_external_source",
    50: "unlikely",
    60: "implausible",
    70: "unfavorable_measurement_context",  # present
    80: "insufficient_database",
    90: "doubt_from_device_source",  # present
    100: "doubt_from_user",
}

In [ ]:
[generation_type_map[int(k)] for k in sorted(df_complete.generation.dropna().unique())]

In [ ]:
[
    trustworthiness_type_map[int(k)]
    for k in sorted(df_complete.trustworthiness.dropna().unique())
]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Create a 2D histogram of generation vs type, normalized by type, including NA values

# First, let's handle NaN values by replacing them with a specific label
df_plot = df_complete.copy()
df_plot["generation_labeled"] = df_plot["generation"].fillna(-1)  # Use -1 for NA values
df_plot["type_clean"] = df_plot["type"].fillna("NA")

# Map generation values to readable labels
generation_labels = generation_type_map.copy()
generation_labels[-1] = "NA"  # Add NA label

# Create the cross-tabulation for the heatmap
crosstab = pd.crosstab(
    df_plot["type_clean"], df_plot["generation_labeled"], normalize="index"
)

# Map column names to readable labels
crosstab.columns = [
    generation_labels.get(col, f"Unknown_{col}") for col in crosstab.columns
]

# Convert to percentages for display
crosstab_percent = crosstab * 100

# Create the plot with percentages
plt.figure(figsize=(12, 8))
sns.heatmap(
    crosstab_percent,
    annot=True,
    fmt=".1f",
    cmap="inferno_r",
    cbar_kws={"label": "Percentage (normalized by type)"},
)

plt.title(
    "2D Histogram: Density of Generation Types by Data Type\n(Normalized by Type, Including NA values)"
)
plt.xlabel("Generation Type")
plt.ylabel("Data Type")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Also create a regular count heatmap for reference
plt.figure(figsize=(12, 8))
crosstab_counts = pd.crosstab(df_plot["type_clean"], df_plot["generation_labeled"])
crosstab_counts.columns = [
    generation_labels.get(col, f"Unknown_{col}") for col in crosstab_counts.columns
]

sns.heatmap(
    crosstab_counts, annot=True, fmt="_d", cmap="viridis_r", cbar_kws={"label": "Count"}
)

plt.title(
    "2D Histogram: Count of Generation Types by Data Type\n(Raw Counts, Including NA values)"
)
plt.xlabel("Generation Type")
plt.ylabel("Data Type")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Print some summary statistics
print("Summary of Generation Types by Data Type (Percentages):")
print(crosstab.round(1))
print(f"\nTotal records: {len(df_complete):,}")
print(f"Records with NA generation: {df_complete['generation'].isna().sum():,}")
print(f"Records with NA type: {df_complete['type'].isna().sum():,}")

In [ ]:
result = df_complete.groupby("generation")["type"].unique()
for generation, types in result.items():
    generation_name = generation_type_map.get(generation, f"Unknown ({generation})")
    print(f"{generation_name} ({generation}):")
    for type_name in types:
        print(f"  - {type_name}")
    print()

## Create (customer, time) -> timezoneOffset map

In [ ]:
# df = df_complete # TODO change it later to df_backup_recent, but keep the other cols in create_backup.py first
df = df_backup_recent
df["startTimestamp_day"] = df["startTimestamp"].dt.floor("D")


In [ ]:
# don't need the values for the timezones
df.drop(
    columns=["stringValue", "doubleValue", "longValue", "booleanValue"], inplace=True
)

In [ ]:
# types_relevant_for_tz = [
#     "Latitude",
#     "Longitude",
#     "ActivityType",
#     "ActivityTypeDetail1",
#     "ActivityTypeDetail2",
#     "RunBinary",
#     "ActiveBurnedCalories",
#     # "Steps",
#     # "CoveredDistance",
#     "HeartRate",
#     "ActiveBinary",
#     # "SleepAwakeBinary",
#     # "SleepBinary",
#     # "SleepStateBinary",
#     # "SleepLightBinary",
#     # "SleepDeepBinary",
#     "BikeBinary",
#     "WalkBinary",
#     # "ElevationGain",
#     # "FloorsClimbed",
#     # "BloodPressureDiastolic",
#     # "BloodPressureSystolic",
#     "SPO2",
#     # "AtrialFibrillationDetection",
#     # "RawECGVoltage",
#     # "Weight",
#     # "SleepInBedBinary",
#     # "SleepREMBinary",
#     # "Height",
#     # "BodyTemperature",
#     # "FatFreeMass",
#     # "FatMass",
#     # "MuscleMass",
#     # "BoneMass",
#     # "FatRatio",
#     # "VO2max",
#     # "RespirationRateSleep",
#     # "SnoringBinary",
#     # "Rmssd",
#     # "PulseWaveVelocity",
# ]


# df = df[df["type"].isin(types_relevant_for_tz)].copy().reset_index(drop=True)
# df["type"] = df["type"].cat.remove_unused_categories()

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
df["createdAt_day"] = df["createdAt"].dt.floor("D")

In [ ]:
tz_series = df.groupby(["customer", "startTimestamp_day"], observed=True)[
    "timezoneOffset"
].apply(lambda x: x.dropna().unique())

In [ ]:
df_tz = tz_series.reset_index(name="tzs")
df_tz["n_tzs"] = df_tz["tzs"].apply(len)
df_tz

In [ ]:
# Create date range from min to max year in df, string make it interpreted as year
date_range = pd.date_range(
    f"{df['startTimestamp_day'].min().year}",
    f"{df['startTimestamp_day'].max().year + 1}",
    freq="h",
    tz="Europe/Berlin",
)

berlin_offsets = pd.Series(date_range.map(lambda x: x.utcoffset()), index=date_range)
print("Offsets shape:", berlin_offsets.shape)
print("Unique offsets:", berlin_offsets.unique())

# Find where timezone offsets change
offset_changes = berlin_offsets != berlin_offsets.shift(1)
dst_changes = date_range[offset_changes]
dst_changes = dst_changes[
    1:
]  # remove the first value, which is always different than shift (nan)
print("EU Timezone changes:", dst_changes)

In [ ]:
dst_changes.date

In [ ]:
df_tz["dst_berlin_change_day"] = df_tz.startTimestamp_day.dt.date.isin(dst_changes.date)
df_tz

In [ ]:
df_tz[(df_tz.n_tzs > 1) & (~df_tz.dst_berlin_change_day)][
    "customer"
].nunique()  # problematic days

In [ ]:
df_tz

In [ ]:
df.groupby("type", observed=False)["timezoneOffset"].unique()

In [ ]:
# Create pivot table with types as rows, timezoneOffsets as columns, and count of unique customers
pivot_table = df.pivot_table(
    values="customer",
    index="type",
    columns="timezoneOffset",
    aggfunc="nunique",
    fill_value=0,
)

print("Pivot table: Types (rows) × Timezone Offsets (columns) = Unique customer counts")
pivot_table

In [ ]:
# Create heatmap visualization of the pivot table using seaborn
plt.figure(figsize=(12, 8))

# Calculate max value excluding timezone offsets 60 and 120 for better color scaling
excluded_tz = [60, 120]
pivot_excluded = pivot_table.drop(
    columns=[tz for tz in excluded_tz if tz in pivot_table.columns]
)
vmax = (
    pivot_excluded.values.max()
    if not pivot_excluded.empty
    else pivot_table.values.max()
)

# Create annotation matrix to hide zeros
annot_matrix = pivot_table.copy()
annot_matrix = annot_matrix.where(annot_matrix != 0, "")

# Create seaborn heatmap
sns.heatmap(
    pivot_table,
    annot=annot_matrix,  # Show values in cells (but hide zeros)
    fmt="",  # Format as string (since we're using custom annotations)
    cmap="viridis",  # Color palette
    vmax=vmax,  # Set max value for color scale
    cbar_kws={"label": "Number of Unique Customers"},
    linewidths=0.5,  # Add grid lines
    linecolor="white",
)  # Grid line color

# Add labels and title
plt.xlabel("Timezone Offset (minutes)")
plt.ylabel("Data Type")
plt.title(
    "Number of Unique Customers by Data Type and Timezone Offset\n(Color scale adjusted excluding TZ 60 & 120)"
)

# Rotate x-axis labels for better readability
plt.xticks(rotation=45)
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
df_coverage = (
    df.groupby(["type", "customer"])["startTimestamp_day"].nunique()
    / df.groupby("customer")["startTimestamp_day"].nunique()
)
df_coverage = df_coverage.reset_index(name="coverage")

In [ ]:
# Create a descriptive statistics plot for coverage by data type
plt.figure(figsize=(15, 8))

# Get the transposed descriptive statistics
coverage_stats = (
    df_coverage.groupby("type")["coverage"].describe().drop("count", axis=1)
)

# Create a heatmap of the statistics
sns.heatmap(
    coverage_stats,
    annot=True,
    fmt=".3f",
    cmap="viridis",
    vmax=1.0,
    vmin=0.0,
    cbar_kws={"label": "proportion of days with at least one entry"},
    linewidths=0.5,
)

plt.title("Coverage Statistics by Data Type over customers")
plt.xlabel("Data Type")
plt.ylabel("Statistical Measures")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Also display the numerical table
print("Coverage Statistics Summary (Transposed):")
print(coverage_stats.round(4))

In [ ]:
df_tz[df_tz.n_tzs == 1]["tzs"].apply(lambda x: x[0]).value_counts().sort_index()

In [ ]:
df

In [ ]:
df_tz[(df_tz.n_tzs > 1) & ~(df_tz.dst_berlin_change_day)]  # problems are here

In [ ]:
# df["generation"] = df["generation"].fillna(-1)
# df["trustworthiness"] = df["trustworthiness"].fillna(-1)

# generation_type_map = {
#     -1: "unknown",
#     10: "manual_entry",
#     20: "manual_measurement",
#     30: "automated_measurement",
#     40: "calculation",
#     50: "smartphone",  # not present
#     60: "tracker",
#     70: "third_party",
# }
# df["generation_type"] = df.generation.map(generation_type_map).astype("category")

# trustworthiness_type_map = {
#     -1: "unknown",
#     10: "plausible",
#     20: "verified_from_device_source",  # present
#     30: "verified_from_user",
#     40: "verified_from_external_source",
#     50: "unlikely",
#     60: "implausible",
#     70: "unfavorable_measurement_context",  # present
#     80: "insufficient_database",
#     90: "doubt_from_device_source",  # present
#     100: "doubt_from_user",
# }
# df["trustworthiness_type"] = df.trustworthiness.map(trustworthiness_type_map).astype(
#     "category"
# )

In [ ]:
df["created_after_ndays"] = (df["createdAt"] - df["startTimestamp"]).dt.days

In [ ]:
df.head(10)

In [ ]:
# df[df.type == "ActivityTypeDetail1"].longValue.value_counts()

In [ ]:
from matplotlib import ticker

df_problematic = df_tz[
    (df_tz.n_tzs > 1) & ~(df_tz.dst_berlin_change_day)
]  # problems are here

c = df_problematic["customer"].unique()[18]
# c = "0xWn"
# c = "0ePW"
# c = "5V5c"
# c = "S5sH"
# c = "NGlo"

# weird ones
no_tz_customers = ["3oNs", "9nSQ", "INjr", "LEgz", "SssR", "Zv6E"]
c = no_tz_customers[0]

df_problematic_c = df_problematic[df_problematic.customer == c]
day_c_min = df_problematic_c.startTimestamp_day.min()
day_c_max = df_problematic_c.startTimestamp_day.max()


df_cd = df[
    (df["customer"] == c)
    & (df["startTimestamp_day"] >= day_c_min - pd.Timedelta(days=14))
    & (df["startTimestamp_day"] <= day_c_max + pd.Timedelta(days=60))
]

# Add jitter to the timezoneOffset column
jitter_strength = 10  # Adjust this value to control the amount of jitter
df_cd = df_cd.copy()  # Avoid modifying the original DataFrame
df_cd["timezoneOffset_jittered"] = df_cd["timezoneOffset"] + np.random.uniform(
    -jitter_strength, jitter_strength, size=len(df_cd)
)

g = sns.relplot(
    data=df_cd,
    x="startTimestamp",
    # x="createdAt",
    # row="type",
    # y="timezoneOffset",
    y="timezoneOffset_jittered",
    # hue = "generation_type",
    # hue = "source",
    # hue = "trustworthiness_type",
    # hue = "createdAt_day",
    # hue = "created_after_ndays",
    height=1.5,
    aspect=8,
    s=10,
    palette="flare",
    # palette="crest",
    kind="scatter",
)

g.axes.flat[0].yaxis.set_major_locator(ticker.MultipleLocator(120))
g.axes.flat[0].yaxis.set_minor_locator(ticker.MultipleLocator(60))

# Mark days that have timezone conflicts for this customer - span across all subplots
conflict_days = df_problematic_c["startTimestamp_day"].tolist()
for day in conflict_days:
    day_start = pd.Timestamp(day)
    day_end = day_start + pd.Timedelta(hours=24)
    for ax in g.axes.flat:
        ax.axvspan(day_start, day_end, color="gray", alpha=0.2)
g.figure.suptitle(
    f"Customer: {c}, Date Range: {day_c_min.date()} to {day_c_max.date()}"
)

plt.tight_layout()
plt.show()


### df tz gps

In [ ]:
df_tz_gps = (
    df[df["type"].isin(["Latitude", "Longitude"])]
    .groupby(["customer", "startTimestamp_day"], observed=True)["timezoneOffset"]
    .apply(lambda x: x.unique())
    .reset_index(name="tzs_gps")
)

In [ ]:
df_tz_gps

In [ ]:
df_tz

In [ ]:
df_tz = df_tz.merge(df_tz_gps, on=["customer", "startTimestamp_day"], how="left")
df_tz.head(10)

In [ ]:
df_tz["gps_exists"] = ~df_tz["tzs_gps"].isna()
df_tz[(df_tz.n_tzs > 1) & ~(df_tz.dst_berlin_change_day)]["gps_exists"].value_counts()
# -> solves ~50-60% of the problematic cases

In [ ]:
df.head()

In [ ]:
temp = df[df.type.isin(["Latitude", "Longitude"])]
((temp["createdAt"] - temp["startTimestamp"]).dt.total_seconds() / 60 / 60).describe(
    [0.5, 0.75, 0.9, 0.95, 0.99]
)

### df tz createdAt

In [ ]:
df_tz_createdat = (
    df.groupby(["customer", "createdAt_day"], observed=True)["timezoneOffset"]
    .apply(lambda x: x.unique())
    .reset_index(name="tzs_createdat")
)

df_tz_createdat.head(10)

In [ ]:
df[df["type"].isin(["ActivityTypeDetail1", "ActivityTypeDetail2"])][
    "createdAt"
].isna().sum() / df[df["type"].isin(["ActivityTypeDetail1", "ActivityTypeDetail2"])][
    "createdAt"
].dropna().count()

In [ ]:
df.head()

In [ ]:
df_tz_activitydetailcreatedat = (
    df[df["type"].isin(["ActivityTypeDetail1", "ActivityTypeDetail2"])]
    .groupby(
        ["customer", "createdAt_day"],
        observed=True,
    )["timezoneOffset"]
    .apply(lambda x: x.unique())
    .reset_index(name="tzs_activitydetailcreatedat")
)


In [ ]:
df_tz_activitydetailcreatedat.rename(columns={"createdAt_day": "asdfasdfasdf"})

In [ ]:
df_tz_activitydetailcreatedat

In [ ]:
df_tz = df_tz.merge(
    df_tz_activitydetailcreatedat.rename(
        columns={"createdAt_day": "startTimestamp_day"}
    ),
    on=["customer", "startTimestamp_day"],
    how="left",
)

In [ ]:
df_tz

In [ ]:
df_tz[~df_tz.gps_exists].tzs_gps

In [ ]:
# df_tz[df_tz.gps_exists]["tzs_gps"].isna().sum()
df_tz["tzs_gps"].isna().sum()

In [ ]:
df_tz[df_tz.gps_exists]["tzs_gps"].value_counts()

In [ ]:
df_tz.tzs_gps

In [ ]:
df_tz.tzs_gps.dropna().apply(lambda x: len(x))

In [ ]:
# df_tz.gps_exists.value_counts()
df_tz

# return_tz

In [ ]:
df_tz["return_tz"] = pd.NA  # we want to return this
df_tz["return_source"] = pd.NA  # where did the tz come from

In [ ]:
df_tz.tzs.isna().sum()  # TODO, there might be a problem here if we don't have the date in the df

In [ ]:
df_tz.return_tz[50]

In [ ]:
mask_gps = df_tz.gps_exists & (df_tz.tzs_gps.dropna().apply(len) == 1)
df_tz.loc[mask_gps, "return_tz"] = df_tz.tzs_gps[mask_gps].apply(lambda x: x[0])
df_tz.loc[mask_gps, "return_source"] = "gps_single"

In [ ]:
mask_activitydetailcreatedat = (~mask_gps) & (
    df_tz.tzs_activitydetailcreatedat.dropna().apply(len) == 1
)
df_tz.loc[mask_activitydetailcreatedat, "return_tz"] = (
    df_tz.tzs_activitydetailcreatedat[mask_activitydetailcreatedat].apply(
        lambda x: x[0]
    )
)
df_tz.loc[mask_activitydetailcreatedat, "return_source"] = (
    "activitydetailcreatedat_single"
)

In [ ]:
df_tz

In [ ]:
df_tz.return_tz.isna().value_counts()

In [ ]:
df_tz.startTimestamp_day.dt.month

In [ ]:
# for the dst, assume there is the new timezone, in Berlin it's gonna be just one hour different
# (1->2, and 2->1; instead of 2->3 & 3->2, as it shoud be) (since we convert at the 00:00 UTC )
# If the primary metric would be the time from previous midnight (as in the sleep),
# then it should be the other way round
# TODO move it to the end
change2summer = df_tz.dst_berlin_change_day & (df_tz.startTimestamp_day.dt.month == 3)
change2winter = df_tz.dst_berlin_change_day & (df_tz.startTimestamp_day.dt.month == 10)
dst_summer_mask = change2summer & df_tz.return_tz.isin([60, 120])
dst_winter_mask = change2winter & df_tz.return_tz.isin([60, 120])
df_tz.loc[dst_summer_mask, "return_tz"] = 120
df_tz.loc[dst_winter_mask, "return_tz"] = 60

# df_tz.loc[dst_summer_mask, "return_source"] += "_dst_adjusted"
# df_tz.loc[dst_winter_mask, "return_source"] += "_dst_adjusted"

In [ ]:
df_tz[
    df_tz.tzs.apply(lambda x: any(val not in [60, 120] for val in x if pd.notna(val)))
]

In [ ]:
# df_tz[df_tz.tzs_gps.notna() & df_tz.tzs_gps.dropna().apply(lambda x: len(x) > 1)]
# df_tz[df_tz.tzs_activitydetailcreatedat.notna() & df_tz.tzs_activitydetailcreatedat.dropna().apply(lambda x: len(x) > 1)]
df_tz[
    df_tz.tzs_gps.notna()
    & df_tz.tzs_gps.dropna().apply(lambda x: len(x) > 1)
    & df_tz.return_tz.isna()
]

In [ ]:
# # prev_available = \

# # row = df_tz.iloc[22507]
# row = df_tz.iloc[0]
# row.startTimestamp_day
# # Safe approach using empty DataFrame check
# row = df_tz.iloc[749]
# # rows with multiple gps tzs and no return_tz
# for ind, row in df_tz[
#     df_tz.tzs_gps.notna()
#     & df_tz.tzs_gps.dropna().apply(lambda x: len(x) > 1)
#     & df_tz.return_tz.isna()
# ].iterrows():
#     # print(row)
#     prev_df = df_tz[
#         (df_tz.customer == row.customer)
#         & (df_tz.startTimestamp_day < row.startTimestamp_day)
#         & (df_tz.return_tz.notna())
#     ].sort_values("startTimestamp_day")
#     prev_available = prev_df.iloc[-1] if not prev_df.empty else None

#     next_df = df_tz[
#         (df_tz.customer == row.customer)
#         & (df_tz.startTimestamp_day > row.startTimestamp_day)
#         & (df_tz.return_tz.notna())
#     ].sort_values("startTimestamp_day")
#     next_available = next_df.iloc[0] if not next_df.empty else None

#     prev_available, row, next_available

#     potential_tzs = row.tzs_gps
#     assert potential_tzs is not None and len(potential_tzs) > 1
#     if prev_available is None and next_available is None:
#         inferred_tz = None
#         source = "no_previous_no_next"
#         print(source)
#         logging.error("This edge case :"
#             f"{source} for customer {row.customer} at {row.startTimestamp_day}"
#                       f" with potential tzs {potential_tzs}"
#                       )
#     elif prev_available is None and next_available is not None:
#         if next_available.return_tz in potential_tzs:
#             inferred_tz = next_available.return_tz
#             source = "no_previous_next_is_fine"
#         else:
#             inferred_tz = None
#             source = "no_previous_conflict_with_next"
#     elif prev_available is not None and next_available is None:
#         if prev_available.return_tz in potential_tzs:
#             inferred_tz = prev_available.return_tz
#             source = "previous_is_fine_no_next"
#         else:
#             inferred_tz = None
#             source = "conflict_with_previous_no_next"
#             print(source, " : ")
#             print(prev_available.return_tz)
#             print(potential_tzs)

#     # both previous and next available
#     elif prev_available is not None and next_available is not None:
#         if prev_available.return_tz not in potential_tzs:
#             if next_available.return_tz in potential_tzs:
#                 inferred_tz = next_available.return_tz
#                 source = "conflict_with_previous_next_is_fine"
#             else:
#                 inferred_tz = None
#                 source = "conflict_with_both"
#                 print(source, " : ")
#                 print(prev_available.return_tz, next_available.return_tz)
#         else:
#             if next_available.return_tz not in potential_tzs:
#                 inferred_tz = prev_available.return_tz
#                 source = "conflict_with_next_previous_is_fine"
#             else:
#                 # Both previous and next are in potential tzs
#                 # Choose the closer one in time
#                 days_to_prev = (
#                     row.startTimestamp_day - prev_available.startTimestamp_day
#                 ).days
#                 days_to_next = (
#                     next_available.startTimestamp_day - row.startTimestamp_day
#                 ).days

#                 if days_to_prev < days_to_next:
#                     inferred_tz = prev_available.return_tz
#                     source = f"inferred_from_previous_{days_to_prev}d"
#                 elif days_to_next < days_to_prev:
#                     inferred_tz = next_available.return_tz
#                     source = f"inferred_from_next_{days_to_next}d"
#                 else:  # Equal distance, prefer previous
#                     inferred_tz = prev_available.return_tz
#                     source = f"inferred_from_previous_{days_to_prev}d_equal_dist"
#     if inferred_tz is not None:
#         df_tz.at[ind, "return_tz"] = inferred_tz
#         df_tz.at[ind, "return_source"] = source
#     print(source)


In [ ]:
def infer_timezone_from_neighbors(row, df_tz, potential_tzs=None):
    """Infer timezone based on previous and next available data points."""
    # ! it assumes certain structure of row and df_tz, and that row is from df_tz

    # TODO if memory constraint is a problem, extract df_tz_customer first
    # Get previous and next available data points
    prev_df = df_tz[
        (df_tz.customer == row.customer)
        & (df_tz.startTimestamp_day < row.startTimestamp_day)
        & (df_tz.return_tz.notna())
    ].sort_values("startTimestamp_day")
    prev_available = prev_df.iloc[-1] if not prev_df.empty else None

    next_df = df_tz[
        (df_tz.customer == row.customer)
        & (df_tz.startTimestamp_day > row.startTimestamp_day)
        & (df_tz.return_tz.notna())
    ].sort_values("startTimestamp_day")
    next_available = next_df.iloc[0] if not next_df.empty else None

    # Check which neighbors are valid (exist and match potential timezones)
    if potential_tzs is None:
        prev_valid = prev_available is not None
        next_valid = next_available is not None
    else:
        prev_valid = (
            prev_available is not None and prev_available.return_tz in potential_tzs
        )
        next_valid = (
            next_available is not None and next_available.return_tz in potential_tzs
        )

    # Handle cases based on validity
    if not prev_valid and not next_valid:
        if prev_available is None and next_available is None:
            logging.warning(
                f"No neighbors for customer {row.customer} at {row.startTimestamp_day} with tzs {potential_tzs}"
            )
            return None, "no_previous_no_next"
        else:
            return (
                None,
                "no_previous_conflict_with_next"
                if prev_available is None
                else (
                    "conflict_with_previous_no_next"
                    if next_available is None
                    else "conflict_with_both"
                ),
            )

    # Choose the valid neighbor (or closer one if both valid)
    if prev_valid and not next_valid:
        return (
            prev_available.return_tz,
            "previous_is_fine_no_next"
            if next_available is None
            else "conflict_with_next_previous_is_fine",
        )

    if next_valid and not prev_valid:
        return (
            next_available.return_tz,
            "no_previous_next_is_fine"
            if prev_available is None
            else "conflict_with_previous_next_is_fine",
        )

    if prev_available.return_tz == next_available.return_tz:
        return prev_available.return_tz, "both_neighbors_agree"

    # Both valid - choose closer one
    days_to_prev = (row.startTimestamp_day - prev_available.startTimestamp_day).days
    days_to_next = (next_available.startTimestamp_day - row.startTimestamp_day).days

    if days_to_prev <= days_to_next:  # Prefer previous on tie
        return prev_available.return_tz, f"inferred_from_previous_{days_to_prev}d" + (
            "_equal_dist" if days_to_prev == days_to_next else ""
        )
    else:
        return next_available.return_tz, f"inferred_from_next_{days_to_next}d"

In [ ]:
df_tz_before = df_tz.copy()
for ind, row in df_tz[
    df_tz.tzs_gps.dropna().apply(lambda x: len(x) > 1) & df_tz.return_tz.isna()
].iterrows():
    inferred_tz, source = infer_timezone_from_neighbors(row, df_tz_before, row.tzs_gps)
    source = "gps_multiple_" + source
    df_tz.at[ind, "return_source"] = source
    if inferred_tz is not None:
        df_tz.at[ind, "return_tz"] = inferred_tz
    else:
        logging.info(
            f"Could not infer tz for customer {row.customer} at {row.startTimestamp_day} with potential tzs {row.tzs_gps} - {source}"
        )

In [ ]:
df_tz.return_source.value_counts()

In [ ]:
df_tz_before = df_tz.copy()
for ind, row in df_tz[
    df_tz.tzs_activitydetailcreatedat.dropna().apply(lambda x: len(x) > 1)
    & df_tz.return_tz.isna()
].iterrows():
    potential_tzs = row.tzs_activitydetailcreatedat
    inferred_tz, source = infer_timezone_from_neighbors(
        row, df_tz_before, potential_tzs
    )
    source = "activitydetail_multiple_" + source
    df_tz.at[ind, "return_source"] = source
    if inferred_tz is not None:
        df_tz.at[ind, "return_tz"] = inferred_tz
    else:
        logging.info(
            f"Could not infer tz for customer {row.customer} at {row.startTimestamp_day} with potential tzs {potential_tzs} - {source}"
        )

In [ ]:
df_tz[
    df_tz.tzs_activitydetailcreatedat.dropna().apply(lambda x: len(x) > 1)
    & df_tz.return_tz.isna()
]


In [ ]:
df_tz.return_source.isna().value_counts()

In [ ]:
df_tz_before = df_tz.copy()
for ind, row in df_tz[df_tz.return_tz.isna()].iterrows():
    inferred_tz, source = infer_timezone_from_neighbors(
        row, df_tz_before, potential_tzs=None
    )
    source = "interpolate_" + source
    df_tz.at[ind, "return_source"] = source
    if inferred_tz is not None:
        df_tz.at[ind, "return_tz"] = inferred_tz
    else:
        logging.info(
            f"Could not infer tz for customer {row.customer} at {row.startTimestamp_day} - {source}"
        )

In [ ]:
# df_tz.return_source.value_counts()
df_tz.return_tz.isna().value_counts()

In [ ]:
df_tz[df_tz.return_tz.isna()].customer.unique()

In [ ]:
# df[df.customer == '3oNs'] # only one day, no gps
# df[df.customer == '9nSQ'] # only for one day, just one ECG recording
# df[df.customer == 'INjr'] # only two days, two ECG sessions and that's it
# df[df.customer == 'LEgz'] # only 10 days, mostly running sessions
# df[df.customer == 'SssR'] # data for one month 2023-05 -- 2023-06, no gps activitytypedetails without createdAt
# df[df.customer == 'Zv6E'] # data for two months 2023-09 -- 2023-11, no gps, activitytypedetails without createdAt

# i guess we can assume the Berlin timezone for them, but do they even provide enought data for analysis?


In [ ]:
# fill the other with berlin timezone
still_missing_mask = df_tz.return_tz.isna()
if still_missing_mask.any():
    df_tz.loc[still_missing_mask, "return_tz"] = (
        df_tz.loc[still_missing_mask, "startTimestamp_day"] + pd.Timedelta(hours=12)
    ).dt.tz_convert("Europe/Berlin").map(lambda x: x.utcoffset()).dt.total_seconds() / 60
    df_tz.loc[still_missing_mask, "return_source"] = "assumed_berlin"

In [ ]:
# adjust again for dst changes
# for the dst, assume there is the new timezone, in Berlin it's gonna be just one hour different
# (1->2, and 2->1; instead of 2->3 & 3->2, as it shoud be) (since we convert at the 00:00 UTC )
# If the primary metric would be the time from previous midnight (as in the sleep),
# then it should be the other way round
# TODO move it to the end
change2summer = df_tz.dst_berlin_change_day & (df_tz.startTimestamp_day.dt.month == 3)
change2winter = df_tz.dst_berlin_change_day & (df_tz.startTimestamp_day.dt.month == 10)
dst_summer_mask = change2summer & df_tz.return_tz.isin([60, 120])
dst_winter_mask = change2winter & df_tz.return_tz.isin([60, 120])
df_tz.loc[dst_summer_mask, "return_tz"] = 120
df_tz.loc[dst_winter_mask, "return_tz"] = 60

df_tz.loc[dst_summer_mask, "return_source"] += "_dst_adjusted"
df_tz.loc[dst_winter_mask, "return_source"] += "_dst_adjusted"

In [ ]:
#? TODO the case when there's no data for particular day, but we still need it for EMA/ECG

In [ ]:
df_tz.return_source.value_counts()